# ANÁLISE EXPLORATÓRIA

### IMPORT DAS BIBLIOTECAS E DOS DADOS

In [ ]:
import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import zarr
import gcsfs

print("zarr:", zarr.__version__)
print("xarray:", xr.__version__)

In [ ]:
url = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

In [ ]:
ds = xr.open_zarr(url,storage_options={"token": "anon"},)
ds

In [ ]:
#print(ds)
#print(ds.dims)
#print(ds.coords)
print(list(ds.data_vars))

### PRECITAÇÃO

In [ ]:
#nomes relacionados a precitação
[var for var in ds.data_vars if "precip" in var.lower()]

In [ ]:
precip = ds["total_precipitation"]

precip

In [ ]:
print(precip.dims)
print(precip.attrs)

In [ ]:
def extrair_precipitacao_mensal(
    ds,
    inicio,
    fim,
    lat_norte=15,
    lat_sul=-60,
    lon_oeste=275,
    lon_leste=330,
):
    precip = ds["total_precipitation"].sel(
        time=slice(inicio, fim),
        latitude=slice(lat_norte, lat_sul),
        longitude=slice(lon_oeste, lon_leste),
    )

    precip_mensal_mm = precip.sum(dim="time") * 1000

    #precipitação não pode ser negativa.
    #pequenos valores negativos podem surgir por precisão numérica.
    precip_mensal_mm = precip_mensal_mm.clip(min=0)

    precip_mensal_mm.attrs["units"] = "mm"
    precip_mensal_mm.attrs["long_name"] = "Monthly total precipitation"

    return precip_mensal_mm

In [ ]:
jan_2023 = extrair_precipitacao_mensal(
    ds,
    "2023-01-01",
    "2023-01-31T23:00:00"
)

jan_2023.plot(figsize=(10, 8))
plt.title("Precipitação mensal - Janeiro/2023 - América do Sul")
plt.show()
#salvando as visualizações
jan_2023.to_netcdf("../data/interim/precip_jan_2023_america_sul.nc")

In [ ]:
fev_2023 = extrair_precipitacao_mensal(
    ds,
    "2023-02-01",
    "2023-02-28T23:00:00"
)

fev_2023.plot(figsize=(10, 8))
plt.title("Precipitação mensal - Fevereiro/2023 - América do Sul")
plt.show()

fev_2023.to_netcdf("../data/interim/precip_fev_2023_america_sul.nc")

In [ ]:
print("Mínimo:", float(fev_2023.min()))
print("Máximo:", float(fev_2023.max()))
print("Média:", float(fev_2023.mean()))

print("Quantidade de valores negativos:", int((fev_2023 < 0).sum()))

### TEMPERATURA

In [ ]:
#nomes relacionados a temperatura
[var for var in ds.data_vars if "temperature" in var.lower()]

### VENTO

In [ ]:
#nomes relacionados ao vento
[var for var in ds.data_vars if "wind" in var.lower()]

### LATITUDE E LONGITUDE

In [ ]:
print(ds.latitude.min().values)
print(ds.latitude.max().values)

In [ ]:
print(ds.longitude.min().values)
print(ds.longitude.max().values)

## TEMPO (MIN, MAX)

In [ ]:
print(ds.time.min().values)
print(ds.time.max().values)